In [3]:
import pandas as pd
import numpy as np
import os

filepath = "data/gene expression/3_GEOexpression.txt"  # adjust if needed

# ── 0. File size ────────────────────────────────────────────────────────────
size_mb = os.path.getsize(filepath) / (1024 ** 2)
print(f"=== FILE SIZE: {size_mb:.1f} MB ===")

# ── 1. Sniff first 5 lines raw — detect separator and structure ─────────────
print("\n=== RAW FIRST 5 LINES ===")
with open(filepath, "r") as f:
    for i, line in enumerate(f):
        print(repr(line[:400]))
        if i == 4:
            break

# ── 2. Detect separator ─────────────────────────────────────────────────────
import csv
with open(filepath, "r") as f:
    sample = f.read(2000)
dialect = csv.Sniffer().sniff(sample)
sep = dialect.delimiter
print(f"\n=== DETECTED SEPARATOR: {repr(sep)} ===")

# ── 3. Read header + 5 rows only ────────────────────────────────────────────
print("\n=== HEADER + 5 ROWS (memory safe) ===")
df_head = pd.read_csv(filepath, sep=sep, nrows=5, low_memory=False)
print(f"  Columns detected: {df_head.shape[1]}")
print(f"  Column names (first 10): {df_head.columns[:10].tolist()}")
print(f"  Column names (last 5):   {df_head.columns[-5:].tolist()}")
print(df_head.iloc[:, :8].to_string())

# ── 4. Count rows without loading full file ─────────────────────────────────
print("\n=== ROW COUNT (line count minus header) ===")
with open(filepath, "r") as f:
    row_count = sum(1 for _ in f) - 1
print(f"  Total data rows: {row_count:,}")

# ── 5. Column count and dtype sniff from header alone ───────────────────────
print("\n=== COLUMN OVERVIEW ===")
cols = df_head.columns.tolist()
print(f"  Total columns: {len(cols)}")

# Check what the columns look like — are they cell line names, GSM IDs, etc.
has_gsm  = sum(1 for c in cols if str(c).startswith("GSM"))
has_ach  = sum(1 for c in cols if str(c).startswith("ACH"))
has_ensg = sum(1 for c in cols if str(c).startswith("ENSG"))
print(f"  Columns starting GSM:  {has_gsm}")
print(f"  Columns starting ACH:  {has_ach}")
print(f"  Columns starting ENSG: {has_ensg}")

# ── 6. ID format in first column ────────────────────────────────────────────
print("\n=== FIRST COLUMN ID FORMAT ===")
first_col = cols[0]
print(f"  First column name: '{first_col}'")
sample_vals = df_head[first_col].astype(str).tolist()
print(f"  Sample values: {sample_vals}")
print(f"  ENSG format: {any(v.startswith('ENSG') for v in sample_vals)}")
print(f"  Gene symbol: {any(v.isalpha() for v in sample_vals)}")

# ── 7. Read a chunked sample to assess missingness and value range ───────────
# Read first 10,000 rows only — enough to characterise without OOM
print("\n=== CHUNKED SAMPLE (first 10,000 rows) ===")
df_sample = pd.read_csv(filepath, sep=sep, nrows=10000, low_memory=False)

numeric_cols = df_sample.select_dtypes(include='number').columns
print(f"  Numeric columns in sample: {len(numeric_cols)}")

if len(numeric_cols) > 0:
    vals = df_sample[numeric_cols].values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"  Sample value range: {vals.min():.3f} – {vals.max():.3f}")
    print(f"  Sample mean: {vals.mean():.3f}  median: {np.median(vals):.3f}")
    if vals.max() < 25:
        print("  → Suggests log2-transformed")
    elif vals.max() < 1000:
        print("  → Suggests TPM or normalised counts")
    else:
        print("  → Suggests raw counts")

    # Missingness in sample
    row_null = df_sample[numeric_cols].isnull().mean(axis=1)
    col_null = df_sample[numeric_cols].isnull().mean(axis=0)
    print(f"\n  Missingness in 10k-row sample:")
    print(f"    Mean % missing per row:    {row_null.mean()*100:.1f}%")
    print(f"    Rows with >50% missing:    {(row_null > 0.5).sum():,}")
    print(f"    Mean % missing per col:    {col_null.mean()*100:.1f}%")
    print(f"    Cols with >50% missing:    {(col_null > 0.5).sum():,}")
    print(f"    Cols with 0% missing:      {(col_null == 0).sum():,}")

# ── 8. GEO metadata columns check ───────────────────────────────────────────
print("\n=== GEO METADATA CHECK ===")
meta_candidates = [c for c in cols if not str(c).startswith("GSM")
                   and df_sample[c].dtype == object]
print(f"  Non-numeric / metadata columns: {meta_candidates}")
for col in meta_candidates[:5]:
    print(f"    '{col}' sample values: {df_sample[col].dropna().unique()[:5].tolist()}")

# ── 9. Study structure check ─────────────────────────────────────────────────
# GEO data might have a study/GSE column indicating which study each row/col came from
print("\n=== STUDY STRUCTURE ===")
for col in cols[:15]:
    col_str = str(col)
    if "GSE" in col_str or "study" in col_str.lower() or "series" in col_str.lower():
        print(f"  Possible study column: '{col}'")
        print(f"  Unique values: {df_sample[col].nunique()}")
        print(f"  Examples: {df_sample[col].dropna().unique()[:5].tolist()}")

print("\n  Note: if no GSE column found, study membership may be encoded")
print("  in column names (e.g. GSM IDs map to GSE via File 10)")

=== FILE SIZE: 1046.4 MB ===

=== RAW FIRST 5 LINES ===
'Gene\tGSM101610\tGSM101615\tGSM101616\tGSM101667\tGSM101668\tGSM101671\tGSM101672\tGSM101673\tGSM101674\tGSM101675\tGSM101676\tGSM101677\tGSM101678\tGSM101679\tGSM101680\tGSM101685\tGSM101686\tGSM101687\tGSM101688\tGSM1017454\tGSM1017455\tGSM1017456\tGSM1017457\tGSM1017458\tGSM1017459\tGSM1017460\tGSM1017461\tGSM1017462\tGSM1017463\tGSM1017464\tGSM1017465\tGSM1017466\tGSM1017467\tGSM1017468\tGSM1017469\tGSM1017470\tGSM1017471\tGSM1017'
'ENSG00000000003\t33.6156997680664\t553.249755859375\t540.452209472656\t599.43115234375\t625.242736816406\t400.554656982422\t412.99560546875\t427.403900146484\t461.12353515625\t921.422546386719\t608.151916503906\t847.329711914062\t904.734558105469\t392.945098876953\t407.066528320312\t875.932739257812\t542.902770996094\t340.507690429688\t368.763885498047\t245.801788330078\t268.549957275391\t271.404327392578\t823.704162597'
'ENSG00000000005\t40.9256820678711\t31.3274059295654\t33.9349670410156\t34.21

In [8]:
import pandas as pd
import numpy as np

# ── 1. Sniff File 10 raw first ──────────────────────────────────────────────
print("=== FILE 10 RAW FIRST 5 LINES ===")
with open("data/nomenclature/10_GEOInfo.txt", "r") as f:
    for i, line in enumerate(f):
        print(repr(line[:400]))
        if i == 4:
            break

import csv
with open("data/nomenclature/10_GEOInfo.txt", "r") as f:
    sample = f.read(2000)
dialect = csv.Sniffer().sniff(sample)
sep10 = dialect.delimiter
print(f"\n=== FILE 10 SEPARATOR: {repr(sep10)} ===")

geo_info = pd.read_csv("data/nomenclature/10_GEOInfo.txt", sep=sep10, low_memory=False)
print(f"\nShape: {geo_info.shape}")
print(f"Columns: {geo_info.columns.tolist()}")
print(geo_info.head(5).to_string())

# ── 2. Column breakdown ─────────────────────────────────────────────────────
print("\n=== FILE 10 COLUMN DETAIL ===")
for col in geo_info.columns:
    n_unique = geo_info[col].nunique()
    null_count = geo_info[col].isna().sum()
    sample_vals = geo_info[col].dropna().astype(str).unique()[:3].tolist()
    has_gse = any(v.startswith("GSE") for v in sample_vals)
    has_gsm = any(v.startswith("GSM") for v in sample_vals)
    has_ach = any(v.startswith("ACH") for v in sample_vals)
    print(f"  '{col}': {n_unique} unique, {null_count} nulls  "
          f"GSE={has_gse} GSM={has_gsm} ACH={has_ach}  examples={sample_vals}")

# ── 3. GSM overlap between File 3 and File 10 ──────────────────────────────
print("\n=== GSM OVERLAP: File 3 vs File 10 ===")
with open("data/gene expression/3_GEOexpression.txt", "r") as f:
    gsm_cols = [c for c in f.readline().strip().split("\t") if c.startswith("GSM")]
print(f"  GSM IDs in expression file (File 3): {len(gsm_cols):,}")

# Find the GSM column in File 10
gsm_col_10 = next((c for c in geo_info.columns
                   if geo_info[c].astype(str).str.startswith("GSM").any()), None)
print(f"  GSM column in File 10: '{gsm_col_10}'")

if gsm_col_10:
    gsm_in_info = set(geo_info[gsm_col_10].dropna().astype(str))
    gsm_in_expr = set(gsm_cols)
    print(f"  GSM IDs in File 10:                  {len(gsm_in_info):,}")
    print(f"  Overlap:                             {len(gsm_in_expr & gsm_in_info):,}")
    print(f"  In expression, not in info:          {len(gsm_in_expr - gsm_in_info):,}")
    print(f"  In info, not in expression:          {len(gsm_in_info - gsm_in_expr):,}")

# ── 4. Unique cell lines and studies ────────────────────────────────────────
print("\n=== UNIQUE CELL LINES & STUDIES ===")
cell_line_col = next((c for c in geo_info.columns
                      if any(x in c.lower() for x in
                             ["cell", "line", "name", "title", "sample"])), None)
gse_col = next((c for c in geo_info.columns
                if geo_info[c].astype(str).str.startswith("GSE").any()), None)

print(f"  Likely cell line column: '{cell_line_col}'")
print(f"  Likely GSE/study column: '{gse_col}'")

if cell_line_col:
    print(f"  Unique cell lines: {geo_info[cell_line_col].nunique():,}")
    print(f"  Sample values: {geo_info[cell_line_col].dropna().unique()[:8].tolist()}")

if gse_col:
    print(f"  Unique GSE studies: {geo_info[gse_col].nunique():,}")
    print(f"  GSE IDs: {geo_info[gse_col].dropna().unique().tolist()}")

# ── 5. Replicate structure ──────────────────────────────────────────────────
print("\n=== REPLICATE STRUCTURE ===")
if cell_line_col and gsm_col_10:
    samples_per_line = geo_info.groupby(cell_line_col)[gsm_col_10].count()
    print(f"  Unique cell lines:              {len(samples_per_line):,}")
    print(f"  Mean GSM samples per line:      {samples_per_line.mean():.1f}")
    print(f"  Median GSM samples per line:    {samples_per_line.median():.1f}")
    print(f"  Max GSM samples per line:       {samples_per_line.max()}")
    print(f"  Cell lines with 1 sample:       {(samples_per_line == 1).sum():,}")
    print(f"  Cell lines with 2-3 samples:    {((samples_per_line >= 2) & (samples_per_line <= 3)).sum():,}")
    print(f"  Cell lines with >3 samples:     {(samples_per_line > 3).sum():,}")

if gse_col and gsm_col_10:
    print(f"\n  Samples per GSE study:")
    print(geo_info.groupby(gse_col)[gsm_col_10].count().sort_values(ascending=False).to_string())

# ── 6. Per-sample sum — normalisation check ─────────────────────────────────
print("\n=== VALUE DISTRIBUTION & NORMALISATION CHECK ===")
df_chunk = pd.read_csv(
    "data/gene expression/3_GEOexpression.txt",
    sep="\t", low_memory=False
)
numeric_cols = df_chunk.select_dtypes(include='number').columns

col_sums = df_chunk[numeric_cols].sum(axis=0)
print(f"  Per-sample sum — min:    {col_sums.min():,.0f}")
print(f"  Per-sample sum — max:    {col_sums.max():,.0f}")
print(f"  Per-sample sum — median: {col_sums.median():,.0f}")
print(f"  Per-sample sum — std:    {col_sums.std():,.0f}")
print("  → Median ~1,000,000 = CPM  |  widely variable = raw counts  |  ~uniform low = TPM")

zero_pct = (df_chunk[numeric_cols] == 0).mean().mean() * 100
print(f"\n  Zero values: {zero_pct:.1f}% of all measurements")

# ── 7. Gene overlap with HPA ────────────────────────────────────────────────
print("\n=== GENE OVERLAP WITH HPA (ENSG format check) ===")
geo_genes = set(df_chunk["Gene"].dropna().astype(str))
print(f"  Genes in GEO file: {len(geo_genes):,}")
hpa_sample = pd.read_csv(
    "data/gene expression/1_4_hpa_rna_celline.tsv",
    sep="\t", usecols=["Gene"], nrows=100000
)
hpa_genes = set(hpa_sample["Gene"].dropna().astype(str))
overlap_genes = geo_genes & hpa_genes
print(f"  HPA gene sample (100k rows): {len(hpa_genes):,} unique")
print(f"  Overlap with GEO: {len(overlap_genes):,}")
print(f"  GEO genes not in HPA sample: {len(geo_genes - hpa_genes):,}")
print(f"  Both use ENSG format: "
      f"{all(g.startswith('ENSG') for g in list(geo_genes)[:20])}")

=== FILE 10 RAW FIRST 5 LINES ===
'Geo_accession\tCEL_file_names\ttitle\tstatus\tsubmission_date\tlast_update_date\ttype\tchannel_count\tsource_name_ch1\torganism_ch1\tcharacteristics_ch1\tplatform_id\tcontact_country\tcontact_institute\tGSE_ID\tGSE_filename\tcell_line\tdisease\torigin\tCellosaurus_ID\tCellline\tMatching_Type\tcell_line_Trimmed\n'
'GSM101610\tGSM101610\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tCVCL_0131\tA-172\tCello GEO GSM\tNA\n'
'GSM101615\tGSM101615\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tCVCL_0393\tLN-229\tCello GEO GSM\tNA\n'
'GSM101616\tGSM101616\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tCVCL_0393\tLN-229\tCello GEO GSM\tNA\n'
'GSM101667\tGSM101667\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tNA\tCVCL_1715\tSW1088\tCello GEO GSM\tNA\n'

=== FILE 10 SEPARATOR: '\t' ===

Shape: (3267, 23)
Columns: ['Geo_accession', 'CEL_file_names', 'title', 'status', 'submission

In [9]:
import pandas as pd

geo_info = pd.read_csv("data/nomenclature/10_GEOInfo.txt", sep="\t", low_memory=False)

# ── 1. True unique cell lines via CVCL ─────────────────────────────────────
print("=== TRUE CELL LINE COUNT VIA CVCL ===")
cvcl_counts = geo_info.groupby("Cellosaurus_ID")["Geo_accession"].count()
print(f"  Unique CVCL IDs:                {len(cvcl_counts):,}")
print(f"  Mean GSM samples per CVCL:      {cvcl_counts.mean():.1f}")
print(f"  Median GSM samples per CVCL:    {cvcl_counts.median():.1f}")
print(f"  Max GSM samples per CVCL:       {cvcl_counts.max()}")
print(f"  CVCLs with 1 sample:            {(cvcl_counts == 1).sum():,}")
print(f"  CVCLs with 2-5 samples:         {((cvcl_counts >= 2) & (cvcl_counts <= 5)).sum():,}")
print(f"  CVCLs with >5 samples:          {(cvcl_counts > 5).sum():,}")
print(f"\n  Top 10 most sampled cell lines:")
print(cvcl_counts.sort_values(ascending=False).head(10).to_string())

# ── 2. CVCL nulls ───────────────────────────────────────────────────────────
print("\n=== CVCL NULL CHECK ===")
null_cvcl = geo_info["Cellosaurus_ID"].isna().sum()
print(f"  Rows with null Cellosaurus_ID: {null_cvcl}")
if null_cvcl > 0:
    print("  These GSMs cannot be mapped to a cell line — drop at ingestion")
    print(geo_info[geo_info["Cellosaurus_ID"].isna()][
        ["Geo_accession", "cell_line", "Cellline"]].head(10).to_string())

# ── 3. Matching_Type breakdown ──────────────────────────────────────────────
print("\n=== MATCHING_TYPE BREAKDOWN ===")
print(geo_info["Matching_Type"].value_counts(dropna=False).to_string())

# ── 4. Cross-study cell line overlap ────────────────────────────────────────
print("\n=== CELL LINES APPEARING IN MULTIPLE STUDIES ===")
cvcl_study = geo_info.dropna(subset=["Cellosaurus_ID", "GSE_ID"])
cvcl_study_count = cvcl_study.groupby("Cellosaurus_ID")["GSE_ID"].nunique()
print(f"  CVCLs in exactly 1 study:   {(cvcl_study_count == 1).sum():,}")
print(f"  CVCLs in 2-3 studies:       {((cvcl_study_count >= 2) & (cvcl_study_count <= 3)).sum():,}")
print(f"  CVCLs in >3 studies:        {(cvcl_study_count > 3).sum():,}")
print(f"\n  Top 10 CVCLs across most studies:")
print(cvcl_study_count.sort_values(ascending=False).head(10).to_string())

# ── 5. Overlap with DepMap RNA cell lines ───────────────────────────────────
print("\n=== OVERLAP WITH DEPMAP RNA ===")
profiles = pd.read_csv("data/nomenclature/8_DepMap_OmicsProfiles.csv")
depmap_ach = set(profiles[profiles["Datatype"] == "rna"]["ModelID"].dropna())

sample_info = pd.read_csv("data/nomenclature/9_DepMap_sample_info.csv")
ach_to_cvcl = sample_info.set_index("DepMap_ID")  # use if there's a CVCL col
# Check if sample_info has a CVCL column
print(f"  sample_info columns: {sample_info.columns.tolist()}")

geo_cvcl = set(geo_info["Cellosaurus_ID"].dropna().astype(str))
print(f"  Unique CVCLs in GEO:          {len(geo_cvcl):,}")
# Direct CVCL overlap if sample_info has it
if "CVCL_ID" in sample_info.columns or any("cvcl" in c.lower() for c in sample_info.columns):
    cvcl_col = next(c for c in sample_info.columns if "cvcl" in c.lower())
    depmap_cvcl = set(sample_info[cvcl_col].dropna().astype(str))
    overlap = geo_cvcl & depmap_cvcl
    print(f"  Unique CVCLs in DepMap:       {len(depmap_cvcl):,}")
    print(f"  Overlap GEO ∩ DepMap:         {len(overlap):,}")
else:
    print("  No CVCL column in sample_info — overlap check requires Cellosaurus bridge")

=== TRUE CELL LINE COUNT VIA CVCL ===
  Unique CVCL IDs:                797
  Mean GSM samples per CVCL:      4.0
  Median GSM samples per CVCL:    2.0
  Max GSM samples per CVCL:       478
  CVCLs with 1 sample:            378
  CVCLs with 2-5 samples:         291
  CVCLs with >5 samples:          128

  Top 10 most sampled cell lines:
Cellosaurus_ID
CVCL_7082    478
CVCL_0019     67
CVCL_0031     49
CVCL_0062     41
CVCL_0023     31
CVCL_0179     25
CVCL_0553     24
CVCL_0598     23
CVCL_0531     21
CVCL_0033     20

=== CVCL NULL CHECK ===
  Rows with null Cellosaurus_ID: 108
  These GSMs cannot be mapped to a cell line — drop at ingestion
    Geo_accession cell_line Cellline
236    GSM1230037       ZRT      NaN
237    GSM1230038       ZRT      NaN
238    GSM1230039       ZRT      NaN
239    GSM1230040       ZRT      NaN
240    GSM1230041       ZRT      NaN
241    GSM1230042       ZRT      NaN
242    GSM1230043       ZRT      NaN
243    GSM1230044       ZRT      NaN
865    GSM137441

In [10]:
import pandas as pd

geo_info = pd.read_csv("data/nomenclature/10_GEOInfo.txt", sep="\t", low_memory=False)
sample_info = pd.read_csv("data/nomenclature/9_DepMap_sample_info.csv")

# ── 1. Identify CVCL_7082 and other high-count outliers ────────────────────
print("=== HIGH-SAMPLE CVCL IDENTITY CHECK ===")
top_cvcls = ["CVCL_7082", "CVCL_0019", "CVCL_0031", "CVCL_0062", "CVCL_0023"]
for cvcl in top_cvcls:
    subset = geo_info[geo_info["Cellosaurus_ID"] == cvcl]
    cell_names = subset["Cellline"].dropna().unique().tolist()
    gse_ids = subset["GSE_ID"].dropna().unique().tolist()
    n_gsm = len(subset)
    print(f"  {cvcl}: {n_gsm} GSMs  names={cell_names}  studies={gse_ids}")

# ── 2. RRID column as CVCL bridge ───────────────────────────────────────────
print("\n=== RRID COLUMN IN SAMPLE_INFO ===")
rrid_sample = sample_info["RRID"].dropna().astype(str)
print(f"  Total RRID values:           {len(rrid_sample):,}")
print(f"  Starting with CVCL_:         {rrid_sample.str.startswith('CVCL_').sum():,}")
print(f"  Example values:              {rrid_sample.head(5).tolist()}")

geo_cvcl  = set(geo_info["Cellosaurus_ID"].dropna().astype(str))
depmap_rrid = set(rrid_sample[rrid_sample.str.startswith("CVCL_")])
overlap = geo_cvcl & depmap_rrid

print(f"\n  GEO unique CVCLs:            {len(geo_cvcl):,}")
print(f"  DepMap unique RRIDs (CVCL):  {len(depmap_rrid):,}")
print(f"  Overlap GEO ∩ DepMap:        {len(overlap):,}")
print(f"  GEO CVCLs not in DepMap:     {len(geo_cvcl - depmap_rrid):,}")
print(f"  DepMap CVCLs not in GEO:     {len(depmap_rrid - geo_cvcl):,}")

# ── 3. Resolve top cross-study CVCLs to names ───────────────────────────────
print("\n=== TOP CROSS-STUDY CVCLs RESOLVED ===")
top_multistudy = ["CVCL_0062", "CVCL_1092", "CVCL_0553",
                  "CVCL_0031", "CVCL_1245", "CVCL_1251",
                  "CVCL_1267", "CVCL_0179", "CVCL_0290", "CVCL_0332"]
for cvcl in top_multistudy:
    subset = geo_info[geo_info["Cellosaurus_ID"] == cvcl]
    name = subset["Cellline"].dropna().unique().tolist()
    n_studies = subset["GSE_ID"].dropna().nunique()
    n_gsm = len(subset)
    print(f"  {cvcl}: {name}  studies={n_studies}  GSMs={n_gsm}")

# ── 4. NaN Matching_Type breakdown ──────────────────────────────────────────
print("\n=== NaN MATCHING_TYPE DETAIL ===")
nan_match = geo_info[geo_info["Matching_Type"].isna()]
print(f"  Total NaN Matching_Type rows: {len(nan_match):,}")
print(f"  Of these, null Cellosaurus_ID: {nan_match['Cellosaurus_ID'].isna().sum():,}")
print(f"  Of these, have Cellosaurus_ID: {nan_match['Cellosaurus_ID'].notna().sum():,}")
if nan_match["Cellosaurus_ID"].notna().sum() > 0:
    print("  WARNING — rows with CVCL but no Matching_Type:")
    print(nan_match[nan_match["Cellosaurus_ID"].notna()][
        ["Geo_accession","Cellosaurus_ID","Cellline","cell_line"]].head(5).to_string())

# ── 5. Platform confirmation — microarray not RNA-seq ───────────────────────
print("\n=== PLATFORM CONFIRMATION ===")
print(f"  Unique platform_id values: {geo_info['platform_id'].dropna().unique().tolist()}")
print(f"  GPL570 = Affymetrix Human Genome U133 Plus 2.0 Array (microarray)")
print(f"  Implication: values are RMA-normalised, not TPM")
print(f"  Action: z-score within each GSE study before cross-study aggregation")

# ── 6. Final usable sample count after dropping nulls ───────────────────────
print("\n=== FINAL USABLE SAMPLE COUNT ===")
usable = geo_info[geo_info["Cellosaurus_ID"].notna()]
print(f"  Total GSMs:                  {len(geo_info):,}")
print(f"  Usable GSMs (have CVCL):     {len(usable):,}")
print(f"  Dropped (null CVCL):         {geo_info['Cellosaurus_ID'].isna().sum():,}")
print(f"  Usable unique CVCLs:         {usable['Cellosaurus_ID'].nunique():,}")

=== HIGH-SAMPLE CVCL IDENTITY CHECK ===
  CVCL_7082: 478 GSMs  names=[]  studies=[]
  CVCL_0019: 67 GSMs  names=['SH-SY5Y']  studies=[]
  CVCL_0031: 49 GSMs  names=['MCF7', 'MCF-7']  studies=['GSE41445', 'GSE50811', 'GSE57083', 'GSE10843', 'GSE10890', 'GSE12790', 'GSE34211']
  CVCL_0062: 41 GSMs  names=['MDA-MB-231', 'MDA231', 'MB231']  studies=['GSE41445', 'GSE50811', 'GSE57083', 'GSE65216', 'GSE10843', 'GSE10890', 'GSE12790', 'GSE34211']
  CVCL_0023: 31 GSMs  names=['A549', 'A-549']  studies=['GSE41445', 'GSE57083', 'GSE10843', 'GSE14315', 'GSE34211']

=== RRID COLUMN IN SAMPLE_INFO ===
  Total RRID values:           1,818
  Starting with CVCL_:         1,818
  Example values:              ['CVCL_V607', 'CVCL_0089', 'CVCL_1497', 'CVCL_0993', 'CVCL_WS59']

  GEO unique CVCLs:            797
  DepMap unique RRIDs (CVCL):  1,814
  Overlap GEO ∩ DepMap:        588
  GEO CVCLs not in DepMap:     209
  DepMap CVCLs not in GEO:     1,226

=== TOP CROSS-STUDY CVCLs RESOLVED ===
  CVCL_0062: 